parent: Preselection_v6

In [ ]:
import os, sys
import numpy as np
import pickle
import uproot
import glob
import pandas as pd
from tqdm import tqdm
import json
import ROOT as rt
import fnmatch

## user define
sys.path.append('.')
sys.path.append('./Utils')
from Plotter import HistStack

from beeps import *

# Set config

In [2]:
# ERA="ee17"
# ERA="ee16pre"
# # ERA="mm18"
# ERA="em18"
# # DATE = '20230129_17_eleffmva'
# DATE = '20230301_18'
# # DATE = '20230130_18_dyhighworctight'
# # INPUTDIR = "/eos/user/h/hkwon/Run2/20230130_18_dyhighworctight/mm/"
# INPUTDIR = "/eos/user/h/hkwon/Run2/20230301_18/em/"
import sys
ERA = sys.argv[1]
DATE = sys.argv[2]
INPUTDIR = sys.argv[3]

DUMP_PKL = False
DUMP_SIDEBAND_MET_PKL = False
DUMP_SIDEBAND_MASS_PKL = False
DUMP_SIDEBAND_PT_PKL = False
DUMP_DIVIDED_BY_BTAG = True
# DRAW_SR = True
# RUN_SIDEBAND_MET = False
# RUN_SIDEBAND_MASS = False
# RUN_SIDEBAND_PT = False
# RUN_BTAG = True
# EXPORT_VARIABLE = False

# # # for dev
# ERA = "ee18"
# DATE = "20230622jer_18"
# INPUTDIR = "/eos/user/h/hkwon/Run2/"+DATE+"/ee/"


# Get input

In [3]:
lumi_all={
    "ee16pre":19520,
    "ee16post":16810,
    "ee17":41480,
    "ee18":59830,
    "mm16pre":19520,
    "mm16post":16810,
    "mm17":41480,
    "mm18":59830,    
#     "mm18":14000,    
    "em16pre":19520,
    "em16post":16810,
    "em17":41480,
    "em18":59830     
}

data_evts = {
    "ee16pre":652566543,
    "ee16post":287977113,
    "ee17":460041183,
    "ee18":1326465857,
    "mm16pre":458655573,
    "mm16post":327037312,
    "mm17":739073837,
#     "mm18":944842550,  #949352777 
    "mm18":949352777,  #949352777 
    "em16pre":1111222116,
    "em16post":615014425,
    "em17":1199115020,
#     "em18":2271308407   #2281001366  
    "em18":2281001366   #2281001366  
}

lumi=lumi_all[ERA]
print(ERA, lumi)

channel = ERA.replace("16pre", "")
channel = channel.replace("16post", "")
channel = channel.replace("17", "")
channel = channel.replace("18", "")

def get_df(root_file_name, unneccesary_columns):
    f = uproot.open(root_file_name)
    df = uproot.open(root_file_name)["Events_"+channel].arrays(library="pd")
    df_met_u = uproot.open(root_file_name)["Events_"+channel+"_met_u"].arrays(library="pd")
    df_met_d = uproot.open(root_file_name)["Events_"+channel+"_met_d"].arrays(library="pd")    
    df_met_ue_u = uproot.open(root_file_name)["Events_"+channel+"_met_ue_u"].arrays(library="pd")
    df_met_ue_d = uproot.open(root_file_name)["Events_"+channel+"_met_ue_d"].arrays(library="pd") 
    #     df_scale_u = uproot.open(root_file_name)["Events_"+channel+"_scale_u"].arrays(library="pd")
#     df_scale_d = uproot.open(root_file_name)["Events_"+channel+"_scale_d"].arrays(library="pd")
    info = uproot.open(root_file_name)["Info_"+channel].arrays(library="pd")
#     return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_scale_u.drop(unneccesary_columns, axis=1), df_scale_d.drop(unneccesary_columns, axis=1)
    return sum(info["sumweight"]), df.drop(unneccesary_columns, axis=1), df_met_u.drop(unneccesary_columns, axis=1), df_met_d.drop(unneccesary_columns, axis=1), df_met_ue_u.drop(unneccesary_columns, axis=1), df_met_ue_d.drop(unneccesary_columns, axis=1)

unneccesary_columns = []
dic_input = {
    "ee16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/ee/",
    "mm16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/mm/",
    "em16pre": "/eos/user/h/hkwon/Run2/"+DATE+"_16pre/em/",
    "ee16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/ee/",
    "mm16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/mm/",
    "em16post": "/eos/user/h/hkwon/Run2/"+DATE+"_16post/em/",
    "ee17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/ee/",
    "mm17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/mm/",
    "em17": "/eos/user/h/hkwon/Run2/"+DATE+"_17/em/",
    "ee18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/ee/",
    "mm18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/mm/",    
    "em18": "/eos/user/h/hkwon/Run2/"+DATE+"_18/em/"        
#     "em18": "/eos/user/h/hkwon/Run2/20221120_18/em/"        
}
# input_dir = dic_input[ERA]
input_dir = INPUTDIR
f = []
for (dirpath, dirnames, filenames) in os.walk(input_dir):
    f.extend(filenames)
    break

dic_tup={}
for file in tqdm(f):
    # file naming: flatNtuple_Preselect_WZ1.root, flatNtuple_Preselect_Zp-3700_CH-1345.root
    if not "flatNtuple" in file:
        continue
#     if channel=="mm" and (fnmatch.fnmatch(file, '*E*Run*') or fnmatch.fnmatch(file, '*M*RunB*') or fnmatch.fnmatch(file, '*M*RunC*') or fnmatch.fnmatch(file, '*M*RunD*')): #temp
    if channel=="mm" and fnmatch.fnmatch(file, '*E*Run*'): #temp
        continue
    if channel=="ee" and fnmatch.fnmatch(file, '*M*Run*'):
        continue
#     if channel=="em" and fnmatch.fnmatch(file, '*E*Run*'):
#         continue  
    if "TT" in file and not "hadd" in file:
        continue
    if "DY" in file and not "hadd" in file and not "To" in file:
        continue  
    if "TW" in file and not "hadd" in file:
        continue         
    if "WW" in file and not "hadd" in file:
        continue  
    if "WZ" in file and not "hadd" in file:
        continue      
    if "ZZ" in file and not "hadd" in file:
        continue      
    if "Htt" in file and not "hadd" in file:
        continue  
        
    if "Zp" in file:
        sample=file.split('_')[2]+"_"+file.split('_')[3]
    else:
        print(file)
        sample=file.split('_')[2]
    sample=sample.rstrip('.root')
    try:
        print(sample)
        dic_tup[sample]=get_df(input_dir+file, unneccesary_columns)   
    except:
        print(sample)
        beep()        
        if "Run" in sample: # check file corrupt
            raise    
# dic_tup

ee18 59830


  0%|          | 0/318 [00:00<?, ?it/s]

flatNtuple_Preselect_DY1000To15000.root
DY1000To15000


/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:8

  1%|▏         | 4/318 [00:01<01:35,  3.30it/s]

...

/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:8

100%|█████████▉| 317/318 [03:32<00:01,  1.79s/it]

flatNtuple_Preselect_top_hadd.root
top


/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:805: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[name] = series[name]
/cvmfs/sft.cern.ch/lcg/views/LCG_103swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/uproot/interpretation/library.py:8

100%|██████████| 318/318 [03:34<00:00,  1.48it/s]


In [4]:
beep_ok()

In [5]:
# for key in dic_tup:
#     print(key)

In [6]:
# get x-sec
dic_xsec={}
with open('config/CrossSection.json') as json_file:
    data = json.load(json_file)
    dic_xsec=data
print(dic_xsec)

{'Data': 1, 'DY': 6077.22, 'DY500To700': 0.2558, 'DY700To800': 0.04023, 'DY800To1000': 0.03406, 'DY1000To1500': 0.01828, 'DY1500To2000': 0.002367, 'DY2000To3000': 0.0005409, 'TT': 88.29, 'TTUp': 88.29, 'TTDown': 88.29, 'TThdampUp': 88.29, 'TThdampDown': 88.29, 'TTSemi': 365.34, 'TTZ': 0.2432, 'top': 34.91, 'antitop': 34.91, 'WW': 12.82, 'WZ': 5.23, 'WZQQ': 6.42, 'ZZ': 0.6008, 'WWW': 0.0072, 'WWZ': 0.1707, 'HWW': 0.31, 'H': 2.78, 'Zp-1700_CH-345': 0.03523, 'Zp-1700_CH-595': 0.03523, 'Zp-1700_CH-845': 0.03523, 'Zp-2100_CH-345': 0.01198, 'Zp-2100_CH-595': 0.01198, 'Zp-2100_CH-845': 0.01198, 'Zp-2500_CH-345': 0.004396, 'Zp-2500_CH-595': 0.004396, 'Zp-2500_CH-845': 0.004396, 'Zp-2500_CH-1095': 0.004396, 'Zp-2900_CH-345': 0.001768, 'Zp-2900_CH-595': 0.001768, 'Zp-2900_CH-845': 0.001768, 'Zp-2900_CH-1095': 0.001768, 'Zp-2900_CH-1345': 0.001768, 'Zp-3300_CH-345': 0.000729, 'Zp-3300_CH-595': 0.000729, 'Zp-3300_CH-845': 0.000729, 'Zp-3300_CH-1095': 0.000729, 'Zp-3300_CH-1345': 0.000729, 'Zp-3300

In [7]:
Data_concat, TT_concat, TTSemi_concat, TTZ_concat, DY_concat, top_concat, antitop_concat, WW_concat, WZ_concat, ZZ_concat = ([] for i in range(10))
Data_concat_met_u, TT_concat_met_u, TTSemi_concat_met_u, TTZ_concat_met_u, DY_concat_met_u, top_concat_met_u, antitop_concat_met_u, WW_concat_met_u, WZ_concat_met_u, ZZ_concat_met_u = ([] for i in range(10))
Data_concat_met_d, TT_concat_met_d, TTSemi_concat_met_d, TTZ_concat_met_d, DY_concat_met_d, top_concat_met_d, antitop_concat_met_d, WW_concat_met_d, WZ_concat_met_d, ZZ_concat_met_d = ([] for i in range(10))
Data_concat_scale_u, TT_concat_scale_u, TTSemi_concat_scale_u, TTZ_concat_scale_u, DY_concat_scale_u, top_concat_scale_u, antitop_concat_scale_u, WW_concat_scale_u, WZ_concat_scale_u, ZZ_concat_scale_u = ([] for i in range(10))
Data_concat_scale_d, TT_concat_scale_d, TTSemi_concat_scale_d, TTZ_concat_scale_d, DY_concat_scale_d, top_concat_scale_d, antitop_concat_scale_d, WW_concat_scale_d, WZ_concat_scale_d, ZZ_concat_scale_d = ([] for i in range(10))
TTUp_concat, TTDown_concat, TThdampUp_concat, TThdampDown_concat = ([] for i in range(4)) 
TTUp_concat_met_u, TTDown_concat_met_u, TThdampUp_concat_met_u, TThdampDown_concat_met_u = ([] for i in range(4)) 
TTUp_concat_met_d, TTDown_concat_met_d, TThdampUp_concat_met_d, TThdampDown_concat_met_d = ([] for i in range(4)) 
Data_concat_met_ue_u, Data_concat_met_ue_d = ([] for i in range(2))  

Data_nevts, TT_nevts, TTSemi_nevts, TTZ_nevts, DY_nevts, top_nevts, antitop_nevts, WW_nevts, WZ_nevts, ZZ_nevts = ([] for i in range(10))
TTUp_nevts, TTDown_nevts, TThdampUp_nevts, TThdampDown_nevts = ([] for i in range(4))

merged_dic_df={}
for key in dic_tup:
#     if fnmatch.fnmatch(key, 'DY?') or fnmatch.fnmatch(key, 'DY1?'):
#         print(key)
# #         print(dic_tup[key][0])
#         DY_concat.append(dic_tup[key][1])
#         DY_concat_met_u.append(dic_tup[key][2])
#         DY_concat_met_d.append(dic_tup[key][3])        
# #         DY_concat_scale_u.append(dic_tup[key][4])
# #         DY_concat_scale_d.append(dic_tup[key][5])
#         DY_nevts.append(dic_tup[key][0])        
#     if fnmatch.fnmatch(key, 'TT?') or fnmatch.fnmatch(key, 'TT1?'):
# #         print("TT", key)
# #         print(dic_tup[key][0])
#         TT_concat.append(dic_tup[key][1])
#         TT_concat_met_u.append(dic_tup[key][2])
#         TT_concat_met_d.append(dic_tup[key][3])
# #         TT_concat_scale_u.append(dic_tup[key][4])
# #         TT_concat_scale_d.append(dic_tup[key][5])
#         TT_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TThdampUp?') or fnmatch.fnmatch(key, 'TThdampUp1?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TThdampUp_concat.append(dic_tup[key][1])
#         TThdampUp_concat_met_u.append(dic_tup[key][2])
#         TThdampUp_concat_met_d.append(dic_tup[key][3])
#         TThdampUp_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TThdampDown?') or fnmatch.fnmatch(key, 'TThdampDown1?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TThdampDown_concat.append(dic_tup[key][1])
#         TThdampDown_concat_met_u.append(dic_tup[key][2])
#         TThdampDown_concat_met_d.append(dic_tup[key][3])
#         TThdampDown_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TTUp?') or fnmatch.fnmatch(key, 'TTUp1?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TTUp_concat.append(dic_tup[key][1])
#         TTUp_concat_met_u.append(dic_tup[key][2])
#         TTUp_concat_met_d.append(dic_tup[key][3])
#         TTUp_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TTDown?') or fnmatch.fnmatch(key, 'TTDown1?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TTDown_concat.append(dic_tup[key][1])
#         TTDown_concat_met_u.append(dic_tup[key][2])
#         TTDown_concat_met_d.append(dic_tup[key][3])
#         TTDown_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TTSemi*'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TTSemi_concat.append(dic_tup[key][1])
#         TTSemi_concat_met_u.append(dic_tup[key][2])
#         TTSemi_concat_met_d.append(dic_tup[key][3])
# #         TTSemi_concat_scale_u.append(dic_tup[key][4])
# #         TTSemi_concat_scale_d.append(dic_tup[key][5])
#         TTSemi_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TTZ*'):
# #         print(key)
# #         print(dic_tup[key][0])
#         TTZ_concat.append(dic_tup[key][1])
#         TTZ_concat_met_u.append(dic_tup[key][2])
#         TTZ_concat_met_d.append(dic_tup[key][3])
# #         TTZ_concat_scale_u.append(dic_tup[key][4])
# #         TTZ_concat_scale_d.append(dic_tup[key][5])
#         TTZ_nevts.append(dic_tup[key][0])
#     if fnmatch.fnmatch(key, 'TW?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         top_concat.append(dic_tup[key][1])
#         top_concat_met_u.append(dic_tup[key][2])
#         top_concat_met_d.append(dic_tup[key][3])
# #         top_concat_scale_u.append(dic_tup[key][4])
# #         top_concat_scale_d.append(dic_tup[key][5])
#         top_nevts.append(dic_tup[key][0])        
#     elif fnmatch.fnmatch(key, 'TWa?'):
# #         print(key)
# #         print(dic_tup[key][0])
#         antitop_concat.append(dic_tup[key][1])
#         antitop_concat_met_u.append(dic_tup[key][2])
#         antitop_concat_met_d.append(dic_tup[key][3])
# #         antitop_concat_scale_u.append(dic_tup[key][4])
# #         antitop_concat_scale_d.append(dic_tup[key][5])
#         antitop_nevts.append(dic_tup[key][0]) 
#     if fnmatch.fnmatch(key, 'WW*'):
#         WW_concat.append(dic_tup[key][1])
#         WW_concat_met_u.append(dic_tup[key][2])
#         WW_concat_met_d.append(dic_tup[key][3])
# #         WW_concat_scale_u.append(dic_tup[key][4])
# #         WW_concat_scale_d.append(dic_tup[key][5])
#         WW_nevts.append(dic_tup[key][0])        
#     if fnmatch.fnmatch(key, 'WZ*'):
#         WZ_concat.append(dic_tup[key][1])
#         WZ_concat_met_u.append(dic_tup[key][2])
#         WZ_concat_met_d.append(dic_tup[key][3])
# #         WZ_concat_scale_u.append(dic_tup[key][4])
# #         WZ_concat_scale_d.append(dic_tup[key][5])
#         WZ_nevts.append(dic_tup[key][0])
#     elif fnmatch.fnmatch(key, 'ZZ*'):
#         ZZ_concat.append(dic_tup[key][1])
#         ZZ_concat_met_u.append(dic_tup[key][2])
#         ZZ_concat_met_d.append(dic_tup[key][3])
# #         ZZ_concat_scale_u.append(dic_tup[key][4])
# #         ZZ_concat_scale_d.append(dic_tup[key][5])
#         ZZ_nevts.append(dic_tup[key][0])        
    if fnmatch.fnmatch(key, '*Run*'):
        Data_concat.append(dic_tup[key][1])
        Data_concat_met_u.append(dic_tup[key][2])
        Data_concat_met_d.append(dic_tup[key][3])
        Data_concat_met_ue_u.append(dic_tup[key][4])
        Data_concat_met_ue_d.append(dic_tup[key][5])
#         Data_concat_scale_u.append(dic_tup[key][4])
#         Data_concat_scale_d.append(dic_tup[key][5])
        Data_nevts.append(dic_tup[key][0]) 
#     if fnmatch.fnmatch(key, 'Zp*'): 
#         merged_dic_df[key]=dic_tup[key]
#     if fnmatch.fnmatch(key, 'DY*To*'): 
#         merged_dic_df[key]=dic_tup[key]
# #         print("high", key)
    else: 
        merged_dic_df[key]=dic_tup[key]

In [8]:
dic_tup

{'DY1000To15000': (111767,
       PuppiMET_pre     mass_pre  pt_lead_pre  pt_trail_pre       U_pre  \
  0      149.330338   239.920914   309.276062     43.829586  133.519531   
  1      147.138947  1479.492798   620.409485    547.165344  230.459274   
  2      105.483154   624.740234   323.126434    213.031677   11.601327   
  3      148.780792   407.684448   240.969513    165.043503   80.245865   
  4      451.012329   278.177246   280.446167     49.268917  231.536987   
  ..            ...          ...          ...           ...         ...   
  806    267.142883   337.126709   312.354126     90.402275   45.913597   
  807    532.670898   213.198257   290.626892     42.043556  425.693756   
  808    257.772064   841.302673   575.823669    302.775482   80.064423   
  809    189.251190   401.658417   307.196259     34.167625  438.469940   
  810    247.051620   549.233765   460.762146    163.385345   70.933167   
  
          JZB_pre  dPhill_pre  dPhilMET_pre      MT_pre    MT2_pre  ..

In [9]:
# # free memory 
# dic_tup.clear()

In [10]:
totalinput = """
######################################################
####   Total input number of events (sumweight)   ####
######################################################
"""

print(totalinput)
# if TT_concat:
# #     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d), pd.concat(TT_concat_scale_u), pd.concat(TT_concat_scale_d)
#     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d)
#     tmp = "TT: "+str(sum(TT_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"
# if TThdampUp_concat:
# #     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d), pd.concat(TT_concat_scale_u), pd.concat(TT_concat_scale_d)
#     merged_dic_df['TThdampUp']=sum(TThdampUp_nevts), pd.concat(TThdampUp_concat), pd.concat(TThdampUp_concat_met_u), pd.concat(TThdampUp_concat_met_d)
#     tmp = "TThdampUp: "+str(sum(TThdampUp_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"
# if TThdampDown_concat:
# #     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d), pd.concat(TT_concat_scale_u), pd.concat(TT_concat_scale_d)
#     merged_dic_df['TThdampDown']=sum(TThdampDown_nevts), pd.concat(TThdampDown_concat), pd.concat(TThdampDown_concat_met_u), pd.concat(TThdampDown_concat_met_d)
#     tmp = "TThdampDown: "+str(sum(TThdampDown_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"    
# if TTUp_concat:
# #     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d), pd.concat(TT_concat_scale_u), pd.concat(TT_concat_scale_d)
#     merged_dic_df['TTUp']=sum(TTUp_nevts), pd.concat(TTUp_concat), pd.concat(TTUp_concat_met_u), pd.concat(TTUp_concat_met_d)
#     tmp = "TTUp: "+str(sum(TTUp_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"
# if TTDown_concat:
# #     merged_dic_df['TT']=sum(TT_nevts), pd.concat(TT_concat), pd.concat(TT_concat_met_u), pd.concat(TT_concat_met_d), pd.concat(TT_concat_scale_u), pd.concat(TT_concat_scale_d)
#     merged_dic_df['TTDown']=sum(TTDown_nevts), pd.concat(TTDown_concat), pd.concat(TTDown_concat_met_u), pd.concat(TTDown_concat_met_d)
#     tmp = "TTDown: "+str(sum(TTDown_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
# if TTSemi_concat:
# #     merged_dic_df['TTSemi']=sum(TTSemi_nevts), pd.concat(TTSemi_concat), pd.concat(TTSemi_concat_met_u), pd.concat(TTSemi_concat_met_d), pd.concat(TTSemi_concat_scale_u), pd.concat(TTSemi_concat_scale_d)
#     merged_dic_df['TTSemi']=sum(TTSemi_nevts), pd.concat(TTSemi_concat), pd.concat(TTSemi_concat_met_u), pd.concat(TTSemi_concat_met_d)
#     tmp = "TTSemi: "+str(sum(TTSemi_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"    
# if TTZ_concat:
# #     merged_dic_df['TTZ']=sum(TTZ_nevts), pd.concat(TTZ_concat), pd.concat(TTZ_concat_met_u), pd.concat(TTZ_concat_met_d), pd.concat(TTZ_concat_scale_u), pd.concat(TTZ_concat_scale_d)
#     merged_dic_df['TTZ']=sum(TTZ_nevts), pd.concat(TTZ_concat), pd.concat(TTZ_concat_met_u), pd.concat(TTZ_concat_met_d)
#     tmp = "TTZ: "+str(sum(TTZ_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"
# if DY_concat:
# #     merged_dic_df['DY']=sum(DY_nevts), pd.concat(DY_concat), pd.concat(DY_concat_met_u), pd.concat(DY_concat_met_d), pd.concat(DY_concat_scale_u), pd.concat(DY_concat_scale_d)
#     merged_dic_df['DY']=sum(DY_nevts), pd.concat(DY_concat), pd.concat(DY_concat_met_u), pd.concat(DY_concat_met_d)
#     tmp = "DY: "+str(sum(DY_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n"    
# if top_concat:
#     df=pd.concat(top_concat)
# #     merged_dic_df['top']=sum(top_nevts), pd.concat(top_concat), pd.concat(top_concat_met_u), pd.concat(top_concat_met_d), pd.concat(top_concat_scale_u), pd.concat(top_concat_scale_d)
#     merged_dic_df['top']=sum(top_nevts), pd.concat(top_concat), pd.concat(top_concat_met_u), pd.concat(top_concat_met_d)
#     tmp = "top: "+str(sum(top_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
# if antitop_concat:
#     df=pd.concat(antitop_concat)
# #     merged_dic_df['antitop']=sum(antitop_nevts), pd.concat(antitop_concat), pd.concat(antitop_concat_met_u), pd.concat(antitop_concat_met_d), pd.concat(antitop_concat_scale_u), pd.concat(antitop_concat_scale_d)
#     merged_dic_df['antitop']=sum(antitop_nevts), pd.concat(antitop_concat), pd.concat(antitop_concat_met_u), pd.concat(antitop_concat_met_d)
#     tmp = "antitop: "+str(sum(antitop_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
# if ZZ_concat:
#     df=pd.concat(ZZ_concat)
# #     merged_dic_df['ZZ']=sum(ZZ_nevts), pd.concat(ZZ_concat), pd.concat(ZZ_concat_met_u), pd.concat(ZZ_concat_met_d), pd.concat(ZZ_concat_scale_u), pd.concat(ZZ_concat_scale_d)
#     merged_dic_df['ZZ']=sum(ZZ_nevts), pd.concat(ZZ_concat), pd.concat(ZZ_concat_met_u), pd.concat(ZZ_concat_met_d)
#     tmp = "ZZ: "+str(sum(ZZ_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
# if WZ_concat:
#     df=pd.concat(WZ_concat)
# #     merged_dic_df['WZ']=sum(WZ_nevts), pd.concat(WZ_concat), pd.concat(WZ_concat_met_u), pd.concat(WZ_concat_met_d), pd.concat(WZ_concat_scale_u), pd.concat(WZ_concat_scale_d)
#     merged_dic_df['WZ']=sum(WZ_nevts), pd.concat(WZ_concat), pd.concat(WZ_concat_met_u), pd.concat(WZ_concat_met_d)
#     tmp = "WZ: "+str(sum(WZ_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
# if WW_concat:
#     df=pd.concat(WW_concat)
# #     merged_dic_df['WW']=sum(WW_nevts), pd.concat(WW_concat), pd.concat(WW_concat_met_u), pd.concat(WW_concat_met_d), pd.concat(WW_concat_scale_u), pd.concat(WW_concat_scale_d)
#     merged_dic_df['WW']=sum(WW_nevts), pd.concat(WW_concat), pd.concat(WW_concat_met_u), pd.concat(WW_concat_met_d)
#     tmp = "WW: "+str(sum(WW_nevts))
#     print(tmp)
#     totalinput+=tmp+"\n" 
if Data_concat:
#     merged_dic_df['Data']=sum(Data_nevts), pd.concat(Data_concat), pd.concat(Data_concat_met_u), pd.concat(Data_concat_met_d), pd.concat(Data_concat_scale_u), pd.concat(Data_concat_scale_d)
    merged_dic_df['Data']=sum(Data_nevts), pd.concat(Data_concat), pd.concat(Data_concat_met_u), pd.concat(Data_concat_met_d), pd.concat(Data_concat_met_ue_u), pd.concat(Data_concat_met_ue_d)
    tmp = "Data: "+str(sum(Data_nevts))
    print(tmp)
    totalinput+=tmp+"\n" 
# merged_dic_df["Zp-2500"]=dic_tup["Zp-2500"][0], dic_tup["Zp-2500"][1] 
hadded = ["TT", "TTSemi", "TTUp", "TTDown", "TThdampUp", "TThdampDown", "top", "antitop", "WW", "WZ", "WZQQ", "DY", "TTZ", "ZZ", "H", "HWW", "WWW", "WWZ"]
for p in hadded:
    print(p, ":", merged_dic_df[p][0])


######################################################
####   Total input number of events (sumweight)   ####
######################################################

Data: 1326465857
TT : 143848848
TTSemi : 472557630
TTUp : 56603364
TTDown : 59369412
TThdampUp : 53803428
TThdampDown : 58968886
top : 6877672
antitop : 7748690
WW : 9956710
WZ : 6482815
WZQQ : 17820214
DY : 128601645
TTZ : 9651834
ZZ : 56766176
H : 3750276
HWW : 6374523
WWW : 3480222
WWZ : 5480337


In [11]:
os.system("mkdir -p "+DATE+"/Presel/"+ERA)
with open(DATE+"/Presel/"+ERA+'/Nevents.txt', 'w') as f:
    f.write(totalinput)

In [12]:
print("expected:", data_evts[ERA], ", while: ", sum(Data_nevts))

# try:
#    if data_evts[ERA]!=sum(Data_nevts):
#        raise ValueError('Data events missing!')
# except (ValueError, IndexError):
#    beep()
#    print("expected:", data_evts[ERA], ", while: ", sum(Data_nevts)) 
#    raise

expected: 1326465857 , while:  1326465857


In [13]:
# # temp!!
for key in merged_dic_df:
    df = merged_dic_df[key][1]
    df_met_u = merged_dic_df[key][2]
    df_met_d = merged_dic_df[key][3]    
    df_met_ue_u = merged_dic_df[key][4]
    df_met_ue_d = merged_dic_df[key][5]
#     df_scale_u = merged_dic_df[key][4]
#     df_scale_d = merged_dic_df[key][5]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        df_new = df.drop(i)
        df_met_u_new = df_met_u.drop(i)
        df_met_d_new = df_met_d.drop(i)        
        df_met_ue_u_new = df_met_ue_u
        df_met_ue_d_new = df_met_ue_d
#         print(df_new.isnull().values.any())
        #workaround for tuple assignment
#         merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_scale_u, df_scale_d
        merged_dic_df[key] = merged_dic_df[key][0], df_new, df_met_u_new, df_met_d_new, df_met_ue_u_new, df_met_ue_d_new
        


In [14]:
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre']
sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre', 'trigSF_pre', 'btagSF_pre']
# sys = ["puweight_pre", "l1prefiring_pre", 'effSF_pre', 'topSF_pre', 'genweight_pre']
eventweights = """
####################################################
####       event weights (normalization)        ####
####################################################
"""

def get_dic_with_weight(merged_dic_df):
    dic_df={}
    dic_df_met_u={}
    dic_df_met_d={}
    dic_df_met_ue_u={}
    dic_df_met_ue_d={}
    dic_df_scale_u={}
    dic_df_scale_d={}
    for key, vs in merged_dic_df.items():
        print(key) 
        if vs[0]==0:
            print("missing: ", key, "!, continue without it")
            continue
        if "Data" in key:
            norm=1
        else:
            if fnmatch.fnmatch(key, '*DY*To*'):
                key = key[:len(key)-1]
            norm=lumi*dic_xsec[key]/vs[0]
        print(key, ":", norm)
        global eventweights
        eventweights+=key+":"+str(norm)+"\n"
#         print(eventweights)
        dic_df[key]=vs[1]
        dic_df_met_u[key] = vs[2]
        dic_df_met_d[key] = vs[3]
        dic_df_met_ue_u[key] = vs[4]
        dic_df_met_ue_d[key] = vs[5]
#         dic_df_scale_u[key] = vs[4]
#         dic_df_scale_d[key]=vs[5]
        dic_df[key]["weight"] = vs[1]["genweight_pre"]*norm
        dic_df_met_u[key]["weightmetUp"] = vs[1]["genweight_pre"]*norm
        dic_df_met_d[key]["weightmetDown"] = vs[1]["genweight_pre"]*norm
        dic_df_met_ue_u[key]["weightmetueUp"] = vs[4]["totweight"].copy(deep=True)*norm
        dic_df_met_ue_d[key]["weightmetueDown"] =vs[5]["totweight"].copy(deep=True)*norm
#         dic_df_scale_u[key]["weightscaleUp"] = vs[4]["totweight"].copy(deep=True)
#         dic_df_scale_d[key]["weightscaleDown"] =vs[5]["totweight"].copy(deep=True)
        dic_df[key]["wopu"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wol1"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["woeff"] = vs[1]["genweight_pre"]*norm
        dic_df[key]["wotop"] = vs[1]["genweight_pre"]*norm       
        dic_df[key]["wobsf"] = vs[1]["genweight_pre"]*norm       
      
        for s in sys:
            if s=="genweight_pre":
                continue
            dic_df[key]["weight"] *= vs[1][s]
            dic_df[key]["weighthdampUp"] = dic_df[key]["weight"].copy(deep=True)
            dic_df[key]["weighthdampDown"] = dic_df[key]["weight"].copy(deep=True)
            dic_df[key]["weightPSUp"] = dic_df[key]["weight"].copy(deep=True)
            dic_df[key]["weightPSDown"] = dic_df[key]["weight"].copy(deep=True)
            dic_df_met_u[key]["weightmetUp"] *= vs[1][s]
            dic_df_met_d[key]["weightmetDown"] *= vs[1][s]            

            # n-1 correction
            if s!="puweight_pre":
                dic_df[key]["wopu"] *= vs[1][s]
            if s!="l1prefiring_pre":
                dic_df[key]["wol1"] *= vs[1][s]
            if s!="effSF_pre":
                dic_df[key]["woeff"] *= vs[1][s]
#             if s!="topSF_pre":
            if not "top" in s:
                dic_df[key]["wotop"] *= vs[1][s]
            if s!="btagSF_pre":
                dic_df[key]["wobsf"] *= vs[1][s]
        #btag    
#         dic_df[key]["btag2"] = dic_df[key]["weight"].copy(deep=True)*vs[1]['btagSF2_pre'] 
#         dic_df[key]["btagSF"] = dic_df[key]["weight"].copy(deep=True)*vs[1]['btagSF_pre']
            
        # up, down    
        for s1 in sys:
            #if "top" in s1 or "eff" in s1:
#             if "btag" in s1:
#                 continue
            s_u = s1.replace("_", "_u_")
            s_d = s1.replace("_", "_d_")
            if s1=="genweight_pre":
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_u])
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_d])
            else:
                dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_u]
                dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]             
#     return dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d
    return dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_ue_u, dic_df_met_ue_d

# dic_df, dic_df_met_u, dic_df_met_d, dic_df_scale_u, dic_df_scale_d = get_dic_with_weight(merged_dic_df)
dic_df, dic_df_met_u, dic_df_met_d, dic_df_met_ue_u, dic_df_met_ue_d = get_dic_with_weight(merged_dic_df)


DY1000To15000
DY1000To1500 : 0.009785467982499308
DY1500To20000
DY1500To2000 : 0.0012113801686825314
DY2000To30000
DY2000To3000 : 0.00028492232043809756
DY500To7000
DY500To700 : 0.09659806229684098
DY700To8000
DY700To800 : 0.021584576686126282
DY800To10000
DY800To1000 : 0.01815582501781896


/tmp/ipykernel_713/2417925856.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"] = vs[1]["genweight_pre"]*norm
/tmp/ipykernel_713/2417925856.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["wopu"] = vs[1]["genweight_pre"]*norm
/tmp/ipykernel_713/2417925856.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-

DY
DY : 2.8273360935624114


/tmp/ipykernel_713/2417925856.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]
/tmp/ipykernel_713/2417925856.py:91: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_u]
/tmp/ipykernel_713/2417925856.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

/tmp/ipykernel_713/2417925856.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_u])
/tmp/ipykernel_713/2417925856.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)*np.fabs(dic_df[key][s_d])
/tmp/ipykernel_713/2417925856.py:91: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

...

/tmp/ipykernel_713/2417925856.py:91: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_u.replace('_u_pre', 'Up')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_u]
/tmp/ipykernel_713/2417925856.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]
/tmp/ipykernel_713/2417925856.py:91: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of 

Zp-4100_CH-1595
Zp-4100_CH-1595 : 5.3847e-05
Zp-4100_CH-1845
Zp-4100_CH-1845 : 5.3847e-05
Zp-4100_CH-345
Zp-4100_CH-345 : 5.3847e-05
Zp-4100_CH-595
Zp-4100_CH-595 : 5.3847e-05


/tmp/ipykernel_713/2417925856.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"+s_d.replace('_d_pre', 'Down')] = dic_df[key]["weight"].copy(deep=True)/dic_df[key][s1]*dic_df[key][s_d]
/tmp/ipykernel_713/2417925856.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["weight"] = vs[1]["genweight_pre"]*norm
/tmp/ipykernel_713/2417925856.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

Zp-4100_CH-845
Zp-4100_CH-845 : 5.3847e-05
antitop
antitop : 0.2695507627740947
top
top : 0.3036878321618129
Data
Data : 1


/tmp/ipykernel_713/2417925856.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["woeff"] = vs[1]["genweight_pre"]*norm
/tmp/ipykernel_713/2417925856.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dic_df[key]["wotop"] = vs[1]["genweight_pre"]*norm
/tmp/ipykernel_713/2417925856.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-

In [15]:
with open(DATE+"/Presel/"+ERA+'/Nevents.txt', 'a') as f:
    f.write(eventweights)

# Preselection

In [16]:
weihtedevents = """
#############################################################
####   unweighted (weighted) #events after preselection  ####
#############################################################
"""
dic_df_raw={}
dic_df_met_u_tmp = {}
dic_df_met_d_tmp = {}
dic_df_met_ue_u_tmp = {}
dic_df_met_ue_d_tmp = {}
# dic_df_scale_u_tmp = {}
# dic_df_scale_d_tmp = {}
# query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and roccoMET_pre>100"
query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"
# query_str="mass_pre>100 and pt_lead_pre>660 and pt_trail_pre>40 and PuppiMET_pre>100"
# query_str="mass_pre>100 and pt_lead_pre>660 and pt_trail_pre>40 and PuppiMET_pre>100"
# query_str="mass_pre>100 and pt_lead_pre>500 and pt_trail_pre>40"
# query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40"
# query_str="mass_pre>100 and pt_lead_pre>660 and pt_trail_pre>40 and MET_pre>100"
# query_str="pt_lead_pre>80 and pt_trail_pre>40"
# query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and MET_pre>80"
# query_str="mass_pre>60"


for key in dic_df:  
    dic_df_raw[key]=dic_df[key].query(query_str)
    dic_df_met_u_tmp[key]=dic_df_met_u[key].query(query_str)
    dic_df_met_d_tmp[key]=dic_df_met_d[key].query(query_str)
    dic_df_met_ue_u_tmp[key]=dic_df_met_ue_u[key].query(query_str) # probably pt cut incorrect
    dic_df_met_ue_d_tmp[key]=dic_df_met_ue_d[key].query(query_str)    
#     dic_df_scale_u_tmp[key]=dic_df_scale_u[key].query(query_str)
#     dic_df_scale_d_tmp[key]=dic_df_scale_d[key].query(query_str)
    unweighted_nevt = dic_df_raw[key].shape[0]
    print(key, ":", unweighted_nevt, "(", sum(dic_df_raw[key]["weight"]), ")")
    print(key, "(met_u) :", dic_df_met_u_tmp[key].shape[0])
    print(key, "(met_d) :", dic_df_met_d_tmp[key].shape[0])
    print(key, "(met_ue_u) :", dic_df_met_ue_u_tmp[key].shape[0])
    print(key, "(met_ue_d) :", dic_df_met_ue_d_tmp[key].shape[0])
#     print(key, "(scale_u) :", dic_df_scale_u_tmp[key].shape[0])
#     print(key, "(scale_d) :", dic_df_scale_d_tmp[key].shape[0])    
#     print(key, "(scale_u) :", dic_df_scale_u[key].shape[0])
#     print(key, "(scale_d) :", dic_df_scale_d[key].shape[0])
    weihtedevents+=key+":"+str(unweighted_nevt)+"("+str(sum(dic_df_raw[key]["weight"]))+")\n"
#     print(key, ":", unweighted_nevt, "(", unweighted_nevt*lumi*dic_xsec[key]/merged_dic_df[key][0], ")")

nested_dic_df={}
nested_dic_df["raw"]=dic_df_raw
nested_dic_df["met_u"]=dic_df_met_u_tmp
nested_dic_df["met_d"]=dic_df_met_d_tmp
nested_dic_df["met_ue_u"]=dic_df_met_ue_u_tmp
nested_dic_df["met_ue_d"]=dic_df_met_ue_d_tmp
# nested_dic_df["scale_u"]=dic_df_scale_u_tmp
# nested_dic_df["scale_d"]=dic_df_scale_d_tmp

DY1000To1500 : 763 ( 3.84241728996858 )
DY1000To1500 (met_u) : 753
DY1000To1500 (met_d) : 692
DY1000To1500 (met_ue_u) : 753
DY1000To1500 (met_ue_d) : 778
DY1500To2000 : 1199 ( 0.7677416762453504 )
DY1500To2000 (met_u) : 1193
DY1500To2000 (met_d) : 1116
DY1500To2000 (met_ue_u) : 1186
DY1500To2000 (met_ue_d) : 1224
DY2000To3000 : 1466 ( 0.22780249032075517 )
DY2000To3000 (met_u) : 1462
DY2000To3000 (met_d) : 1379
DY2000To3000 (met_ue_u) : 1449
DY2000To3000 (met_ue_d) : 1483
DY500To700 : 351 ( 14.149423822760582 )
DY500To700 (met_u) : 350
DY500To700 (met_d) : 318
DY500To700 (met_ue_u) : 350
DY500To700 (met_ue_d) : 379
DY700To800 : 412 ( 4.407401030883193 )
DY700To800 (met_u) : 408
DY700To800 (met_d) : 372
DY700To800 (met_ue_u) : 401
DY700To800 (met_ue_d) : 423
DY800To1000 : 594 ( 5.242474350146949 )
DY800To1000 (met_u) : 592
DY800To1000 (met_d) : 540
DY800To1000 (met_ue_u) : 592
DY800To1000 (met_ue_d) : 625
DY : 274 ( 384.89956068992615 )
DY (met_u) : 272
DY (met_d) : 226
DY (met_ue_u) : 

Zp-4100_CH-345 : 14361 ( 0.743505842587183 )
Zp-4100_CH-345 (met_u) : 14348
Zp-4100_CH-345 (met_d) : 14321
Zp-4100_CH-345 (met_ue_u) : 14363
Zp-4100_CH-345 (met_ue_d) : 14366
Zp-4100_CH-595 : 15431 ( 0.7988277317563188 )
Zp-4100_CH-595 (met_u) : 15420
Zp-4100_CH-595 (met_d) : 15404
Zp-4100_CH-595 (met_ue_u) : 15424
Zp-4100_CH-595 (met_ue_d) : 15420
Zp-4100_CH-845 : 16251 ( 0.8391512919697561 )
Zp-4100_CH-845 (met_u) : 16234
Zp-4100_CH-845 (met_d) : 16221
Zp-4100_CH-845 (met_ue_u) : 16238
Zp-4100_CH-845 (met_ue_d) : 16245
antitop : 2005 ( 493.41323928534985 )
antitop (met_u) : 1997
antitop (met_d) : 1940
antitop (met_ue_u) : 2011
antitop (met_ue_d) : 2013
top : 1675 ( 463.31286592781544 )
top (met_u) : 1672
top (met_d) : 1613
top (met_ue_u) : 1681
top (met_ue_d) : 1706
Data : 10556 ( 10556.0 )
Data (met_u) : 10470
Data (met_d) : 9851
Data (met_ue_u) : 10539
Data (met_ue_d) : 10742


In [17]:
with open(DATE+"/Presel/"+ERA+'/Nevents.txt', 'a') as f:
    f.write(weihtedevents)

In [18]:
if DUMP_PKL or "16" in ERA:
    os.system("mkdir -p "+DATE+"/Presel/"+ERA)
    print("Dump pickle")
    with open(DATE+"/Presel/"+ERA+"/"+"nested_dic_df.pkl", "wb") as f:
        pickle.dump(nested_dic_df, f)

# b tag selection

In [19]:
if DUMP_DIVIDED_BY_BTAG:

    base_line = "mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100"
#     query_str="mass_pre>100 and pt_lead_pre>80 and pt_trail_pre>40 and PuppiMET_pre>100 and njet_pre==1" #temp!
    query_str=base_line+" and nbjetflav_pre==0" #temp!
#     query_str=base_line #temp!

    dic_df_raw={}
    dic_df_met_u_tmp = {}
    dic_df_met_d_tmp = {}
    dic_df_met_ue_u_tmp = {}
    dic_df_met_ue_d_tmp = {}    
    
    print("####################################################")
    print("####   (unweighted) #events after preselection  ####")
    print("####################################################")
    for key in dic_df:  
        dic_df_raw[key]=dic_df[key].query(query_str)
        dic_df_met_u_tmp[key]=dic_df_met_u[key].query(query_str)
        dic_df_met_d_tmp[key]=dic_df_met_d[key].query(query_str)
        dic_df_met_ue_u_tmp[key]=dic_df_met_ue_u[key].query(query_str) # probably pt cut incorrect
        dic_df_met_ue_d_tmp[key]=dic_df_met_ue_d[key].query(query_str)        
        print(key, ":", dic_df_raw[key].shape[0])

    nested_dic_df={}
    nested_dic_df["raw"]=dic_df_raw
    nested_dic_df["met_u"]=dic_df_met_u_tmp
    nested_dic_df["met_d"]=dic_df_met_d_tmp
    nested_dic_df["met_ue_u"]=dic_df_met_ue_u_tmp
    nested_dic_df["met_ue_d"]=dic_df_met_ue_d_tmp

    with open(DATE+"/Presel/"+ERA+"/"+"nested_dic_df_SR1.pkl", "wb") as f:
        pickle.dump(nested_dic_df, f)     
    
    ####################################################
    query_str=base_line+" and nbjetflav_pre>0" #temp!

    dic_df_raw={}
    dic_df_met_u_tmp = {}
    dic_df_met_d_tmp = {}
    dic_df_met_ue_u_tmp = {}
    dic_df_met_ue_d_tmp = {}    
    
    print("####################################################")
    print("####   (unweighted) #events after preselection  ####")
    print("####################################################")
    for key in dic_df:  
        dic_df_raw[key]=dic_df[key].query(query_str)
        dic_df_met_u_tmp[key]=dic_df_met_u[key].query(query_str)
        dic_df_met_d_tmp[key]=dic_df_met_d[key].query(query_str)
        dic_df_met_ue_u_tmp[key]=dic_df_met_ue_u[key].query(query_str) # probably pt cut incorrect
        dic_df_met_ue_d_tmp[key]=dic_df_met_ue_d[key].query(query_str)        
        print(key, ":", dic_df_raw[key].shape[0])

    nested_dic_df={}
    nested_dic_df["raw"]=dic_df_raw
    nested_dic_df["met_u"]=dic_df_met_u_tmp
    nested_dic_df["met_d"]=dic_df_met_d_tmp
    nested_dic_df["met_ue_u"]=dic_df_met_ue_u_tmp
    nested_dic_df["met_ue_d"]=dic_df_met_ue_d_tmp
    
    with open(DATE+"/Presel/"+ERA+"/"+"nested_dic_df_TTCR.pkl", "wb") as f:
        pickle.dump(nested_dic_df, f)    

####################################################
####   (unweighted) #events after preselection  ####
####################################################
DY1000To1500 : 739
DY1500To2000 : 1132
DY2000To3000 : 1398
DY500To700 : 323
DY700To800 : 385
DY800To1000 : 560
DY : 229
HWW : 0
H : 0
TTDown : 25586
TTSemi : 77
TTUp : 24297
TTZ : 4634
TT : 61587
TThdampDown : 25278
TThdampUp : 23486
WWW : 21082
WWZ : 3909
WW : 7086
WZQQ : 229
WZ : 3912
ZZ : 30093
Zp-1700_CH-345 : 8761
Zp-1700_CH-595 : 10563
Zp-1700_CH-845 : 11797
Zp-2100_CH-345 : 10256
Zp-2100_CH-595 : 11924
Zp-2100_CH-845 : 13653
Zp-2500_CH-345 : 11164
Zp-2500_CH-595 : 12939
Zp-2500_CH-845 : 14212
Zp-2900_CH-1095 : 15841
Zp-2900_CH-1345 : 16410
Zp-2900_CH-345 : 12192
Zp-2900_CH-595 : 13645
Zp-2900_CH-845 : 14882
Zp-3300_CH-1095 : 16230
Zp-3300_CH-1345 : 16652
Zp-3300_CH-1595 : 17062
Zp-3300_CH-345 : 13025
Zp-3300_CH-595 : 14113
Zp-3300_CH-845 : 15303
Zp-3700_CH-1095 : 16497
Zp-3700_CH-1345 : 16876
Zp-3700_CH-1595 : 17210
Zp-370